# 17.2 浮點數怎麼放進少量整數格子？


權重可能是0.7或1.2，整數欄位卻只能放0、1、2等數字。直接去掉小數，會把許多不同權重變成同一個0；比較合理的做法是讓整數每一格代表某個浮點間隔。這個間隔叫scale（刻度）。前置是[每種表示占多少bytes](https://birdhackor.github.io/tiny-perceptron-vlm/17.1.html)：int8可用較少空間，但我們還要保存數字如何對應原範圍的規則。

假設scale=0.5，整數1代表0.5，2代表1.0，-1代表-0.5。0.7先除以0.5得到1.4，再取最近整數1；保存1，還原時乘回0.5，得到0.5。原值與還原值差0.2，這是刻度變粗時失去的細節。1.2同理變2.4、取整數2、還原1.0。把浮點數映成少量離散格子的步驟稱quantization（量化）；乘回刻度近似還原稱dequantization（反量化）。

圖的箭頭分別是除刻度、取整數、乘刻度，兩列就是上面兩個數值。注意最右端沒有還原成最左端：保存的整數只告訴我們最近哪個格子，沒有記住原值在格子附近的偏移。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/architecture_quantize_grid.svg")))

In [ ]:
import torch

x = torch.tensor([-0.7, 0.2, 0.7, 1.2])
scale = 0.5
q = (x / scale).round().clamp(-127, 127).to(torch.int8)
restored = q.float() * scale
print("整數格", q.tolist())
print("還原值", restored.tolist())
print("差值", (x - restored).round(decimals=4).tolist())

輸入四個固定浮點數。`round`取最近整數，`clamp`把超出範圍的數字限制在端點；此例沒有碰到端點，但真實低位元欄位範圍有限，不能略過這步。這裡採對稱int8範圍-127至127，雖然int8也能存-128，我們為正負對稱保留一種編碼不用。最後 `to(int8)`真的將每格存成一byte，而不是留在浮點tensor裡假裝已壓縮。

輸出q為 `[-1,0,1,2]`，還原 `[-0.5,0,0.5,1]`，差值 `[-0.2,0.2,0.2,0.2]`。0.2被映成0並非程式錯誤，是因為它更靠近刻度0而非0.5。一般最近格的捨入在未截斷時，誤差不超過半格；超出端點後則可能更大。剛好在兩格中間時，PyTorch round採最近偶數，不是所有情況都往上。

這些差值是便於閱讀的近似十進位寫法。實際`.round(decimals=4).tolist()`仍可能印出`0.20000000298023224`：`round`的結果仍存在有限精度的浮點tensor裡，0.2只能近似表示；`tolist()`轉成普通Python清單時顯示出這個保存值，沒有把每個數字格式化成四位小數。[1.9的浮點數說明](https://birdhackor.github.io/tiny-perceptron-vlm/1.9.html)可幫你區分這種很小的表示尾數與本例約0.2的量化誤差。

練習只把scale改成0.25，先預測整數為 `[-3,1,3,5]`，還原 `[-0.75,0.25,0.75,1.25]`，各項絕對差約0.05，再執行核對。刻度更細能保留細節，但同樣127個正格只覆蓋到31.75，範圍也縮小了；選scale必須一起考慮精度與極值。
